<a href="https://colab.research.google.com/github/Arun-0807/lstm-next-word-prediction/blob/main/Next_word_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Next-Word Prediction

This notebook demonstrates a complete workflow for training a smart keyboard model to predict the next word using a text corpus.

### Project Tasks:
1. **Tokenize the corpus** and construct a vocabulary.
2. **Generate input-target sequences** from the sentences.
3. **Select and justify a sequential model** (LSTM/GRU) and train it.
4. **Evaluate the model** using top-1 or top-5 accuracy.
5. **Demonstrate predictions** for at least five input phrases and discuss the impact of corpus size.

In [11]:
import re
import numpy as np
import tensorflow as tf
from google.colab import files

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split

# Upload your Kaggle text dataset
uploaded = files.upload()

filename = next(iter(uploaded))

with open(filename, "r", encoding="utf-8") as file:
    corpus_text = file.read()

print("Dataset loaded successfully!")
print("Total characters:", len(corpus_text))
print(corpus_text[:500])

Saving 1661-0.txt to 1661-0 (1).txt
Dataset loaded successfully!
Total characters: 581888
﻿
Project Gutenberg's The Adventures of Sherlock Holmes, by Arthur Conan Doyle

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.net


Title: The Adventures of Sherlock Holmes

Author: Arthur Conan Doyle

Release Date: November 29, 2002 [EBook #1661]
Last Updated: May 20, 2019

Language: English

Charac


In [12]:
# Split text into lines
corpus_lines = [
    line.strip()
    for line in corpus_text.splitlines()
    if len(line.strip()) > 10
]

# Select up to 10,000 lines
corpus_lines = corpus_lines[:10000]

# Clean text
clean_sentences = []

for line in corpus_lines:
    line = line.lower()
    line = re.sub(r"[^a-z\s']", "", line)
    line = re.sub(r"\s+", " ", line).strip()

    if len(line.split()) >= 2:
        clean_sentences.append(line)

print("Total sentences/lines:", len(clean_sentences))
print("Sample sentences:")

for sentence in clean_sentences[:5]:
    print(sentence)

Total sentences/lines: 9284
Sample sentences:
project gutenberg's the adventures of sherlock holmes by arthur conan doyle
this ebook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever you may copy it give it away or
reuse it under the terms of the project gutenberg license included
with this ebook or online at wwwgutenbergnet


In [13]:
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(clean_sentences)

total_words = len(tokenizer.word_index) + 1

print("Vocabulary size:", total_words)

Vocabulary size: 8578


In [14]:
input_sequences = []

for sentence in clean_sentences:
    tokens = tokenizer.texts_to_sequences([sentence])[0]

    for i in range(1, len(tokens)):
        input_sequences.append(tokens[:i + 1])

if not input_sequences:
    raise ValueError("No sequences generated. Check your text dataset.")

max_sequence_len = max(len(seq) for seq in input_sequences)

# Prevent excessively long sequences from consuming too much memory
max_sequence_len = min(max_sequence_len, 30)

# Keep only the last 30 tokens of longer sequences
input_sequences = [
    seq[-max_sequence_len:]
    for seq in input_sequences
]

input_sequences = pad_sequences(
    input_sequences,
    maxlen=max_sequence_len,
    padding="pre"
)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("Maximum sequence length:", max_sequence_len)
print("Input shape:", X.shape)
print("Target shape:", y.shape)

Maximum sequence length: 18
Input shape: (97751, 17)
Target shape: (97751,)


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 78200
Testing samples: 19551


In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Embedding(
        input_dim=total_words,
        output_dim=128,
        mask_zero=True
    ),
    LSTM(256),
    Dropout(0.3),
    Dense(total_words, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [25]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/15
1100/1100 ━━━━━━━━━━━━━━━━━━━━ 212s 190ms/step - accuracy: 0.0639 - loss: 6.5566 - val_accuracy: 0.0779 - val_loss: 6.1962
Epoch 2/15
1100/1100 ━━━━━━━━━━━━━━━━━━━━ 200s 182ms/step - accuracy: 0.0965 - loss: 5.8742 - val_accuracy: 0.1086 - val_loss: 5.9115
Epoch 3/15
1100/1100 ━━━━━━━━━━━━━━━━━━━━ 195s 177ms/step - accuracy: 0.1210 - loss: 5.5347 - val_accuracy: 0.1211 - val_loss: 5.8127
Epoch 4/15
1100/1100 ━━━━━━━━━━━━━━━━━━━━ 201s 176ms/step - accuracy: 0.1372 - loss: 5.2633 - val_accuracy: 0.1262 - val_loss: 5.7811
Epoch 5/15
1100/1100 ━━━━━━━━━━━━━━━━━━━━ 202s 176ms/step - accuracy: 0.1508 - loss: 5.0003 - val_accuracy: 0.1321 - val_loss: 5.8095
Epoch 6/15
1100/1100 ━━━━━━━━━━━━━━━━━━━━ 192s 175ms/step - accuracy: 0.1668 - loss: 4.7389 - val_accuracy: 0.1359 - val_loss: 5.8838
Epoch 7/15
1100/1100 ━━━━━━━━━━━━━━━━━━━━ 189s 172ms/step - accuracy: 0.1826 - loss: 4.4812 - val_accuracy: 0.1396 - val_loss: 5.9964


In [17]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

Epoch 1/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 78s 137ms/step - accuracy: 0.0546 - loss: 6.7807 - val_accuracy: 0.0573 - val_loss: 6.4343
Epoch 2/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 73s 134ms/step - accuracy: 0.0697 - loss: 6.1896 - val_accuracy: 0.0765 - val_loss: 6.2797
Epoch 3/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 71s 128ms/step - accuracy: 0.0835 - loss: 5.9349 - val_accuracy: 0.0861 - val_loss: 6.1678
Epoch 4/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 71s 128ms/step - accuracy: 0.0993 - loss: 5.7360 - val_accuracy: 0.1033 - val_loss: 6.0750
Epoch 5/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 70s 127ms/step - accuracy: 0.1143 - loss: 5.5558 - val_accuracy: 0.1134 - val_loss: 6.0160


In [27]:
# Evaluate on the test set
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(f"Test Top-1 Accuracy: {test_accuracy * 100:.2f}%")

# Calculate Top-5 accuracy
probabilities = model.predict(X_test, verbose=0)

top5_indices = np.argsort(
    probabilities,
    axis=1
)[:, -5:]

top5_accuracy = np.mean(
    np.any(top5_indices == y_test[:, None], axis=1)
)

print(f"Test Top-5 Accuracy: {top5_accuracy * 100:.2f}%")

Test Top-1 Accuracy: 13.27%
Test Top-5 Accuracy: 29.99%


In [19]:
def predict_next_word(seed_text, top_k=3):
    tokens = tokenizer.texts_to_sequences([seed_text])[0]

    if not tokens:
        print("No known words in the input.")
        return

    tokens = tokens[-(max_sequence_len - 1):]

    padded = pad_sequences(
        [tokens],
        maxlen=max_sequence_len - 1,
        padding="pre"
    )

    probabilities = model.predict(padded, verbose=0)[0]

    # Exclude the padding token
    probabilities[0] = 0

    top_indices = np.argsort(probabilities)[-top_k:][::-1]

    for index in top_indices:
        word = tokenizer.index_word.get(index, "<UNKNOWN>")
        print(f"{word}: {probabilities[index]:.4f}")

In [20]:
test_phrases = [
    "to be or not to",
    "shall i compare thee to a",
    "the king is",
    "my heart is",
    "romeo and"
]

for phrase in test_phrases:
    print(f"\nInput: {phrase}")

    predict_next_word(phrase, top_k=3)

    print("-" * 30)


Input: to be or not to
be: 0.0668
the: 0.0424
a: 0.0237
------------------------------

Input: shall i compare thee to a
little: 0.0106
man: 0.0101
door: 0.0083
------------------------------

Input: the king is
the: 0.1047
a: 0.0848
his: 0.0305
------------------------------

Input: my heart is
a: 0.0762
the: 0.0510
to: 0.0181
------------------------------

Input: romeo and
the: 0.0513
i: 0.0478
you: 0.0290
------------------------------


In [21]:
!pip install -q ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 49.6 MB/s eta 0:00:00


In [31]:
import numpy as np
import re
import ipywidgets as widgets

from IPython.display import display, clear_output
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Input box
text_input = widgets.Textarea(
    value="my heart is",
    placeholder="Type your sentence here...",
    description="Type here:",
    layout=widgets.Layout(width="95%", height="100px")
)

output = widgets.Output()

def predict_next_words(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s']", "", text).strip()

    if not text:
        return []

    sequence = tokenizer.texts_to_sequences([text])[0]

    if not sequence:
        return []

    sequence = sequence[-(max_sequence_len - 1):]

    padded = pad_sequences(
        [sequence],
        maxlen=max_sequence_len - 1,
        padding="pre"
    )

    probabilities = model.predict(padded, verbose=0)[0].copy()
    probabilities[0] = 0

    top_indices = np.argsort(probabilities)[-3:][::-1]

    predictions = []

    for index in top_indices:
        word = tokenizer.index_word.get(int(index))
        if word:
            predictions.append(
                (word, float(probabilities[index]))
            )

    return predictions


def update_predictions(change=None):
    with output:
        clear_output(wait=True)

        predictions = predict_next_words(text_input.value)

        if not predictions:
            print("Type some words to get predictions.")
            return

        print("Next Word Suggestions\n")

        for word, probability in predictions:
            print(f"{word}: {probability:.2%}")


text_input.observe(update_predictions, names="value")

display(text_input, output)

update_predictions()

Textarea(value='my heart is', description='Type here:', layout=Layout(height='100px', width='95%'), placeholde…

Output()